# E-commerce Analytics: EDA + RFM + KMeans + Recommendations

This notebook uses `online_retail.csv` (Online Retail dataset style) to perform:
- Data cleaning + basic EDA plots
- RFM scoring
- Customer clustering via KMeans
- Collaborative filtering-style recommendations (customer-item similarity)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")


## Load data

Update `CSV_PATH` if needed.

In [ ]:
CSV_PATH = "shopper-spectrum/online_retail.csv"  # relative to this repo root
df = pd.read_csv(CSV_PATH)
df.columns = [c.strip() for c in df.columns]
df.head()

In [ ]:
# Cleaning
required = ["CustomerID", "InvoiceNo", "InvoiceDate", "Quantity", "UnitPrice"]
missing = [c for c in required if c not in df.columns]
missing

In [ ]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")
df = df.dropna(subset=["CustomerID", "InvoiceNo", "InvoiceDate", "Quantity", "UnitPrice"])
df["CustomerID"] = df["CustomerID"].astype(str)

# Remove returns/invalid rows
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] >= 0)].copy()
df["Amount"] = df["Quantity"] * df["UnitPrice"]

df.shape, df["CustomerID"].nunique()

## EDA

In [ ]:
df["month"] = df["InvoiceDate"].dt.to_period("M").astype(str)
rev_by_month = df.groupby("month")["Amount"].sum().sort_index()

plt.figure(figsize=(12,4))
rev_by_month.tail(24).plot(kind="line", marker="o")
plt.xlabel("Month")
plt.ylabel("Revenue (sum Amount)")
plt.xticks(rotation=45, ha="right")
plt.show()

# Top countries
if "Country" in df.columns:
    top_countries = df["Country"].value_counts().head(10)
    plt.figure(figsize=(8,4))
    sns.barplot(x=top_countries.index, y=top_countries.values)
    plt.xticks(rotation=45, ha="right")
    plt.title("Top 10 Countries by order rows")
    plt.show()


In [ ]:
# Top items
item_col = "StockCode" if "StockCode" in df.columns else ("Description" if "Description" in df.columns else None)
item_col

In [ ]:
if item_col is not None:
    top_items = df.groupby(item_col)["Amount"].sum().sort_values(ascending=False).head(15)
    plt.figure(figsize=(10,4))
    sns.barplot(x=top_items.index.astype(str), y=top_items.values)
    plt.xticks(rotation=60, ha="right")
    plt.title("Top 15 Items by Revenue")
    plt.show()
else:
    print("No StockCode/Description column found.")

## RFM scoring

- **Recency**: days since last purchase
- **Frequency**: number of unique invoices
- **Monetary**: total spend

Scores are assigned using quartiles (1..4) for each metric.

In [ ]:
max_date = df["InvoiceDate"].max()
rfm = df.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda s: (max_date - s.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("Amount", "sum"),
).reset_index()

def qcut_score(s, q=4, reverse=False):
    if s.nunique() < q:
        r = s.rank(method="average")
        bins = pd.qcut(r, q=q, labels=False, duplicates="drop")
        out = (bins.astype(float) + 1).astype(int)
    else:
        bins = pd.qcut(s, q=q, labels=False, duplicates="drop")
        out = (bins.astype(float) + 1).astype(int)
    if reverse:
        out = (q + 1 - out)
    return out

rfm["R_Score"] = qcut_score(rfm["Recency"], reverse=True)
rfm["F_Score"] = qcut_score(rfm["Frequency"], reverse=False)
rfm["M_Score"] = qcut_score(rfm["Monetary"], reverse=False)

rfm["RFM_Score"] = rfm["R_Score"].astype(str) + rfm["F_Score"].astype(str) + rfm["M_Score"].astype(str)
rfm["RFM_Score_num"] = rfm["R_Score"] + rfm["F_Score"] + rfm["M_Score"]

rfm.sort_values("RFM_Score_num", ascending=False).head(10)

In [ ]:
plt.figure(figsize=(10,4))
sns.histplot(rfm["RFM_Score_num"], bins=15)
plt.xlabel("R+F+M (1..12)")
plt.ylabel("Customers")
plt.show()

top_rfm = rfm["RFM_Score"].value_counts().head(12)
plt.figure(figsize=(10,4))
sns.barplot(x=top_rfm.index, y=top_rfm.values)
plt.xticks(rotation=45, ha="right")
plt.title("Top RFM segments")
plt.show()

## Clustering with KMeans

In [ ]:
cust = df.groupby("CustomerID").agg(
    recency=("InvoiceDate", lambda s: (max_date - s.max()).days),
    frequency=("InvoiceNo", "nunique"),
    monetary=("Amount", "sum"),
).reset_index()

X = cust[["recency", "frequency", "monetary"]].copy()
scaler = StandardScaler()
Xs = scaler.fit_transform(X)

k = 6
km = KMeans(n_clusters=k, random_state=42, n_init="auto")
cust["cluster"] = km.fit_predict(Xs)

proj = PCA(n_components=2, random_state=42).fit_transform(Xs)
cust["pca1"] = proj[:, 0]
cust["pca2"] = proj[:, 1]

plt.figure(figsize=(10,5))
sns.scatterplot(data=cust, x="pca1", y="pca2", hue="cluster", palette="tab10", s=25)
plt.title("KMeans clusters (PCA projection)")
plt.show()

cust.groupby("cluster").agg(
    customers=("CustomerID", "nunique"),
    recency=("recency", "mean"),
    frequency=("frequency", "mean"),
    monetary=("monetary", "mean"),
).sort_values("monetary", ascending=False)

## Collaborative filtering-style recommendations

This demo builds a customer-item *implicit* matrix (1 if purchased) and recommends items based on Jaccard similarity between customers.

In [ ]:
if item_col is None:
    raise ValueError("No StockCode/Description column found; cannot recommend.")

min_item_support = 25
item_support = df.groupby(item_col)["CustomerID"].nunique()
frequent_items = item_support[item_support >= min_item_support].index

df2 = df[df[item_col].isin(frequent_items)].copy()

pivot = df2.pivot_table(
    index="CustomerID",
    columns=item_col,
    values="InvoiceNo",
    aggfunc="nunique",
    fill_value=0
)
likes = (pivot > 0).astype(np.uint8)

likes.shape

In [ ]:
def recommend_like_jaccard(likes_df, customer_id, top_k=10, min_common=2):
    if customer_id not in likes_df.index:
        return pd.DataFrame({"Item": [], "Score": []})

    target = likes_df.loc[customer_id].values.astype(bool)
    others = likes_df.values.astype(bool)

    intersection = (others & target).sum(axis=1)
    union = (others | target).sum(axis=1)

    sim = np.zeros(likes_df.shape[0], dtype=float)
    mask = union > 0
    sim[mask] = intersection[mask] / union[mask]

    valid = intersection >= min_common
    sim = sim * valid.astype(float)

    # Weighted item popularity among similar customers
    weighted = likes_df.values.T @ sim

    # Exclude items already owned by the target
    already = likes_df.loc[customer_id].astype(bool).values
    weighted[already] = -np.inf

    top_idx = np.argsort(weighted)[::-1][:top_k]
    top_idx = top_idx[np.isfinite(weighted[top_idx])]

    items = likes_df.columns[top_idx]
    scores = weighted[top_idx]

    return pd.DataFrame({"Item": items, "Score": scores}).sort_values("Score", ascending=False).reset_index(drop=True)

# Example
sample_customer = likes.index[0]
recommend_like_jaccard(likes, sample_customer, top_k=10, min_common=2).head(10)